# Rerankers and cross-encoders

A garden centre's staff assistant searches chunked plant-care sheets. The search (a bi-encoder) finds the right
chunk, but often not on top. In this notebook you load the lesson's two real models with sentence-transformers,
search the same 45 care-sheet chunks, rerank the top 10 with a cross-encoder, choose k, time both steps, and
break the reranker with `max_length=128` on purpose.

About 10 minutes. Run the cells from top to bottom. Everything runs on a free CPU runtime, with no account and no key.
The two models are downloaded from the Hugging Face Hub, about 91 MB each.

## 1. Install sentence-transformers

Colab already has PyTorch and transformers. The next cell adds the sentence-transformers version the lesson ran (6.3.0).

**Look for:** no error. pip may print a few lines about other packages; you can ignore them.

In [1]:
%pip install -q sentence-transformers==6.3.0

Note: you may need to restart the kernel to use updated packages.


## 2. The care sheets and the staff questions

Nine plants, five sections each (watering, feeding, pests, winter, problems): 45 chunks. Every chunk starts with
its sheet's title. `QUESTIONS` holds the 20 test questions. The second field is the one chunk that answers
the question, chosen when the question was written.

**Look for:** 45 chunks, 9 plants, 20 questions, and the chunk that answers the orchid question.

In [2]:
SHEETS = {
    # tomato
    "tomato-water": "Tomato: watering. Water little and often so the compost stays evenly moist. Letting pots dry out and then flooding them makes the fruit split and causes blossom end rot.",
    "tomato-feed": "Tomato: feeding. Once the first flowers have set fruit, give a high-potash liquid feed every week until the end of summer.",
    "tomato-pests": "Tomato: pests. Whitefly are small sap-sucking insects that rise in a cloud when the plant is touched. Hang sticky traps and release the parasitic wasp Encarsia once nights stay above 10 °C.",
    "tomato-winter": "Tomato: end of season. Tomatoes are killed by the first frost. Pick the last green fruit in autumn and ripen them indoors next to a banana.",
    "tomato-problems": "Tomato: problems. Blight shows as brown patches on leaves and dark stems after warm, wet weather, and the fruit rots within days. Remove the plants and do not compost them.",
    # lemon
    "lemon-water": "Lemon tree: watering. Water when the top of the compost feels dry, then let the pot drain. Use rainwater if you can. Water much less in winter.",
    "lemon-feed": "Lemon tree: feeding. Use a summer citrus feed from March to September and a winter citrus feed for the rest of the year. Yellow leaves with green veins mean it needs feeding.",
    "lemon-pests": "Lemon tree: pests. Scale insects look like small brown bumps under the leaves and along the stems. They leave a sticky coating on the leaves. Wipe them off with a soft cloth and soapy water.",
    "lemon-winter": "Lemon tree: winter. Citrus are not frost hardy. Bring the tree indoors before the first frost to a cool, bright room, and never leave it outside on a frosty night.",
    "lemon-problems": "Lemon tree: problems. Sudden leaf drop usually follows a move to a new spot, a cold draught or a dry heated room. Keep it in one bright place and mist the leaves.",
    # rose
    "rose-water": "Rose: watering. Water new roses deeply once a week in dry spells for their first summer. Water at the base, not over the leaves.",
    "rose-feed": "Rose: feeding. Feed with a rose fertiliser in spring and again after the first flush of flowers. Mulch with well-rotted manure.",
    "rose-pests": "Rose: pests. Greenfly cluster on soft shoot tips and flower buds in spring. Squash small groups by hand and encourage ladybirds.",
    "rose-winter": "Rose: winter and pruning. Prune bush roses in late winter: cut out dead and crossing stems, then shorten the rest to an outward-facing bud.",
    "rose-problems": "Rose: problems. Black spot is a fungus that marks the leaves with dark blotches; the leaves turn yellow and fall from midsummer. Clear up fallen leaves and choose resistant varieties.",
    # orchid
    "orchid-water": "Moth orchid: watering. Water once the roots in the clear pot turn silvery, usually every week or two. Soak the pot for ten minutes, then let it drain fully.",
    "orchid-feed": "Moth orchid: feeding. Use a weak orchid feed every third watering while it is growing new leaves or flower spikes.",
    "orchid-pests": "Moth orchid: pests. Mealybugs look like bits of white cotton wool in the leaf joints. Dab them with a cotton bud dipped in rubbing alcohol.",
    "orchid-winter": "Moth orchid: winter. Keep it above 15 °C and away from cold windows at night. A drop of a few degrees in autumn helps a new flower spike to start.",
    "orchid-problems": "Moth orchid: problems. Wrinkled, limp leaves with mushy brown roots mean the roots have rotted from too much water. Cut off the dead roots and repot in fresh bark.",
    # hydrangea
    "hydrangea-water": "Hydrangea: watering. Hydrangeas in pots are thirsty. In summer they often wilt in the afternoon heat and recover by evening: water them every day and move the pot out of the midday sun.",
    "hydrangea-feed": "Hydrangea: feeding and colour. Blue flowers need acid soil; in chalky soil they turn pink. A colourant with aluminium sulphate keeps them blue.",
    "hydrangea-pests": "Hydrangea: pests. Vine weevil grubs eat the roots of hydrangeas in pots, and the plant wilts and does not recover even after watering. Water a nematode treatment into the pot in late summer.",
    "hydrangea-winter": "Hydrangea: winter. Leave the dead flower heads on through winter; they protect the buds below from frost. Cut them off in spring.",
    "hydrangea-problems": "Hydrangea: problems. No flowers usually means it was pruned in autumn or spring frost killed the buds. Mophead hydrangeas flower on last year's stems.",
    # dahlia
    "dahlia-water": "Dahlia: watering. Water dahlias generously once a week in dry weather once they are growing strongly. Do not water the tubers before shoots appear.",
    "dahlia-feed": "Dahlia: feeding. Give a high-potash feed every two weeks from midsummer for more flowers. Pinch out the tips of young plants so they bush out.",
    "dahlia-pests": "Dahlia: pests. Earwigs chew ragged holes in dahlia petals at night. Trap them in upturned pots stuffed with straw on top of canes.",
    "dahlia-winter": "Dahlia: winter. After the first frost blackens the leaves, cut the stems back, lift the tubers, dry them upside down and store them in dry compost somewhere frost-free.",
    "dahlia-problems": "Dahlia: problems. Slugs eat young dahlia shoots as soon as they come up in spring. Start tubers in pots and plant out larger plants.",
    # strawberry
    "strawberry-water": "Strawberry: watering. Water at the base in the morning while the fruit is forming. Wet fruit in the evening invites grey mould.",
    "strawberry-feed": "Strawberry: feeding. Feed with a high-potash liquid feed every two weeks once flowers appear.",
    "strawberry-pests": "Strawberry: pests. Birds and slugs take ripening fruit. Net the plants and tuck straw under the fruit to keep it off the soil.",
    "strawberry-winter": "Strawberry: winter. Strawberries are hardy. After fruiting, cut off the old leaves and remove runners you do not want, so the plant rests.",
    "strawberry-problems": "Strawberry: problems. Grey fuzzy mould on the fruit spreads in damp weather. Pick off mouldy fruit at once and space plants for air.",
    # basil
    "basil-water": "Basil: watering. Water basil in the morning and let the top of the compost dry between waterings. Cold, wet roots at night make it collapse.",
    "basil-feed": "Basil: picking. Pinch off the top pairs of leaves often and remove flower buds, so the plant keeps growing leaves.",
    "basil-pests": "Basil: pests. Aphids hide under basil leaves indoors. Rinse the plant under a gentle tap and check again in a few days.",
    "basil-winter": "Basil: winter. Basil is killed by cold below about 10 °C. Grow it on a sunny windowsill indoors from autumn.",
    "basil-problems": "Basil: problems. Black, wilting stems at the base on supermarket pots mean the plants are too crowded and too wet. Split the pot into small clumps.",
    # cucumber
    "cucumber-water": "Cucumber: watering. Keep the compost evenly moist with warm water in the morning. Bitter fruit is a sign of irregular watering.",
    "cucumber-feed": "Cucumber: feeding. Feed every week with a high-potash feed once the first fruits start to swell.",
    "cucumber-pests": "Cucumber: pests. Red spider mite turns leaves pale and speckled with fine webbing in hot, dry glasshouses. Mist the plants and release the predatory mite Phytoseiulus.",
    "cucumber-winter": "Cucumber: end of season. Cucumbers stop cropping as nights cool in autumn. Clear the plants and clean the greenhouse glass before winter.",
    "cucumber-problems": "Cucumber: problems. Powdery mildew is a white dusty coating on leaves, worst in dry soil and still air. Water at the roots and remove the worst leaves.",
}

QUESTIONS = [
    # (question, gold chunk id, kind)
    ("Tiny white bugs fly up when I touch my tomato plants in the greenhouse. How do I get rid of them?", "tomato-pests", "pest"),
    ("Customer says her greenhouse tomatoes keep cracking open just before they ripen. What is she doing wrong?", "tomato-water", "water"),
    ("Brown patches on the tomato leaves after a wet week, and the stems are going dark. What is it?", "tomato-problems", "pest"),
    ("Lady with a lemon tree in the conservatory says the leaves are sticky. What is causing it?", "lemon-pests", "pest"),
    ("Can my lemon tree stay outside when frost is forecast?", "lemon-winter", "frost"),
    ("I moved my lemon tree into the living room and now it is dropping leaves. Is it dying?", "lemon-problems", "water"),
    ("Roses have dark marks on the leaves and the leaves are falling off in July.", "rose-problems", "pest"),
    ("When should I cut back my roses?", "rose-winter", "frost"),
    ("There is white fluffy stuff where the orchid leaves meet the stem.", "orchid-pests", "pest"),
    ("My orchid leaves have gone floppy and wrinkled even though I water it twice a week.", "orchid-problems", "water"),
    ("His hydrangea in a pot wilts every afternoon in the sun but perks up by evening. Is it thirsty?", "hydrangea-water", "water"),
    ("My potted hydrangea wilted and stays wilted even though the compost is wet.", "hydrangea-pests", "pest"),
    ("Should I cut the brown flower heads off my hydrangea before winter?", "hydrangea-winter", "frost"),
    ("Can I leave my dahlias in the ground over winter?", "dahlia-winter", "frost"),
    ("Something is eating holes in my dahlia flowers overnight.", "dahlia-pests", "pest"),
    ("My strawberries are going furry and grey before I can pick them.", "strawberry-problems", "water"),
    ("The basil from the supermarket keeps flopping over and the stems turn black.", "basil-problems", "water"),
    ("Is it too cold to keep basil outside in October?", "basil-winter", "frost"),
    ("My cucumbers taste bitter. Why?", "cucumber-water", "water"),
    ("The cucumber leaves are pale and speckled and there are tiny webs.", "cucumber-pests", "pest"),
]

care_chunks = list(SHEETS.values())
ids = list(SHEETS)
print("chunks:", len(care_chunks), " plants:", len({i.split("-")[0] for i in ids}), " questions:", len(QUESTIONS))
print("orchid-pests:", SHEETS["orchid-pests"])

chunks: 45  plants: 9  questions: 20
orchid-pests: Moth orchid: pests. Mealybugs look like bits of white cotton wool in the leaf joints. Dab them with a cotton bud dipped in rubbing alcohol.


## 3. Load the two models and check them

`search` is the bi-encoder: it turns each text into one vector. `reranker` is the cross-encoder: it reads a
question and a passage together and gives one score. The last lines score the two example pairs from the
cross-encoder's model card.

**Look for:** the model card pairs score about **8.61** and **−4.32**. Then your setup matches the lesson's.
`Identity()` means the scores are raw, not squeezed between 0 and 1. `max_length: 512` is how many tokens of
each pair the reranker reads.

In [3]:
import warnings
warnings.filterwarnings("ignore")
import torch, sentence_transformers, transformers
transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()
torch.manual_seed(0)  # nothing here is random (both models run in eval mode); seeded anyway

from sentence_transformers import SentenceTransformer, CrossEncoder
search = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")
print("sentence-transformers", sentence_transformers.__version__, " torch", torch.__version__)
print("model card example pairs:", reranker.predict([
    ("How many people live in Berlin?", "Berlin had a population of 3,520,031 registered inhabitants in an area of 891.82 square kilometers."),
    ("How many people live in Berlin?", "Berlin is well known for its museums."),
]))
print("reranker activation:", reranker.activation_fn, "max_length:", reranker.max_length)

sentence-transformers 6.3.0  torch 2.15.0
model card example pairs: [ 8.607139  -4.3200784]
reranker activation: Identity() max_length: 512


## 4. Search: the bi-encoder's top 10

The chunk vectors are made once, before any question. For the question, the cell makes one vector, compares it
with every chunk vector (cosine similarity) and sorts.

**Look for:** the four other orchid chunks fill places 1 to 4. `orchid-pests`, the answer, is only **5th**, so it
would not reach the top three that go into the answer.

In [4]:
chunk_vectors = search.encode(care_chunks)
question = "There is white fluffy stuff where the orchid leaves meet the stem."
question_vector = search.encode([question])
scores = search.similarity(question_vector, chunk_vectors)[0]
top_k = scores.argsort(descending=True)[:10].tolist()
print("chunk vectors:", tuple(chunk_vectors.shape))
print("search top 10 (place, chunk id, cosine):")
for place, i in enumerate(top_k, 1):
    print(f"  {place:2}  {ids[i]:<18} {float(scores[i]):.4f}")

chunk vectors: (45, 384)
search top 10 (place, chunk id, cosine):
   1  orchid-feed        0.5170
   2  orchid-problems    0.5155
   3  orchid-water       0.5041
   4  orchid-winter      0.4999
   5  orchid-pests       0.4270
   6  strawberry-water   0.3407
   7  strawberry-pests   0.3277
   8  strawberry-feed    0.3129
   9  dahlia-winter      0.3091
  10  hydrangea-winter   0.2918


## 5. Rerank those 10 with the cross-encoder

`reranker.predict` gets 10 (question, chunk) pairs and runs the model once per pair. Then the cell sorts the
10 candidates by the new score.

**Look for:** every score is below zero, yet the order is right: `orchid-pests` is **1st** with −3.93.
The search score and the reranker score are on different scales; compare scores only by order, within one question.

In [5]:
candidates = [care_chunks[i] for i in top_k]
ce_scores = reranker.predict([(question, c) for c in candidates])
print("reranker order (new place, chunk id, score, place in the search):")
for new_place, j in enumerate(ce_scores.argsort()[::-1], 1):
    print(f"  {new_place:2}  {ids[top_k[j]]:<18} {ce_scores[j]:8.4f}   search place {j + 1}")

reranker order (new place, chunk id, score, place in the search):
   1  orchid-pests        -3.9305   search place 5
   2  orchid-problems     -5.3682   search place 2
   3  dahlia-winter       -8.0354   search place 9
   4  orchid-feed         -8.7559   search place 1
   5  orchid-water        -8.8639   search place 3
   6  orchid-winter       -9.3886   search place 4
   7  hydrangea-winter   -10.3807   search place 10
   8  strawberry-water   -10.9376   search place 6
   9  strawberry-feed    -11.2887   search place 8
  10  strawberry-pests   -11.3217   search place 7


## 6. The whole pipeline, as on the lesson's screen

`find_chunks` is the code from the lesson, line for line: search for `k` candidates, then `reranker.rank` keeps
the best `keep` of them. `rank` returns `corpus_id`, the place in `candidates` counted from 0.

**Look for:** the answer comes back first. The third chunk is `dahlia-winter`, another plant, with −8.04:
`rank` always returns `top_k` chunks, however low they score.

In [6]:
def find_chunks(question, k=10, keep=3):
    question_vector = search.encode([question])
    scores = search.similarity(question_vector, chunk_vectors)[0]
    top_k = scores.argsort(descending=True)[:k].tolist()
    candidates = [care_chunks[i] for i in top_k]
    hits = reranker.rank(question, candidates, top_k=keep, return_documents=True)
    return [hit["text"] for hit in hits]

print("find_chunks(question) returned:")
for text in find_chunks(question):
    print(" ", text[:60] + " ...")
print("reranker.rank(question, candidates, top_k=3):")
for hit in reranker.rank(question, candidates, top_k=3):
    print(" ", {"corpus_id": hit["corpus_id"], "score": round(float(hit["score"]), 4)}, "->", ids[top_k[hit["corpus_id"]]])

find_chunks(question) returned:
  Moth orchid: pests. Mealybugs look like bits of white cotton ...
  Moth orchid: problems. Wrinkled, limp leaves with mushy brow ...
  Dahlia: winter. After the first frost blackens the leaves, c ...
reranker.rank(question, candidates, top_k=3):
  {'corpus_id': 4, 'score': -3.9305} -> orchid-pests
  {'corpus_id': 1, 'score': -5.3682} -> orchid-problems
  {'corpus_id': 8, 'score': -8.0354} -> dahlia-winter


## 7. All 20 questions

For each question the cell scores all 45 chunks with both models once, and keeps the results in `results`.
Then it prints where the answer sits in the search's order, and where it sits after reranking the search's top 10.

**Look for:** the hydrangea "stays wilted" question goes from 3rd to 1st. The tomato "cracking open" question
goes from 2nd to 4th: the reranker can also make a question worse. In the totals: the answer is first for
17 / 20 questions with the search alone and 19 / 20 after reranking. In the top three it is 19 / 20 both times:
the orchid answer moved in, the tomato answer was pushed out.

In [7]:
results = []
for q, gold, kind in QUESTIONS:
    cos = search.similarity(search.encode([q]), chunk_vectors)[0]
    order = [ids[i] for i in cos.argsort(descending=True).tolist()]
    ce = dict(zip(ids, reranker.predict([(q, c) for c in care_chunks]).tolist()))
    results.append({"q": q, "gold": gold, "order": order, "ce": ce})

def place_after(r, k):
    """Place of the answer after reranking the search's top k (None if it is not among them)."""
    reranked = sorted(r["order"][:k], key=lambda c: -r["ce"][c])
    return reranked.index(r["gold"]) + 1 if r["gold"] in reranked else None

print(f"{'answer chunk':<20} search  rerank top 10")
for r in results:
    print(f"{r['gold']:<20} {r['order'].index(r['gold']) + 1:6}  {place_after(r, 10):6}")
search_place = [r["order"].index(r["gold"]) + 1 for r in results]
print("first:      search", sum(p == 1 for p in search_place), "/ 20,  rerank top 10", sum(place_after(r, 10) == 1 for r in results), "/ 20")
print("in top 3:   search", sum(p <= 3 for p in search_place), "/ 20,  rerank top 10", sum(place_after(r, 10) <= 3 for r in results), "/ 20")

answer chunk         search  rerank top 10
tomato-pests              1       1
tomato-water              2       4
tomato-problems           1       1
lemon-pests               1       1
lemon-winter              1       1
lemon-problems            1       1
rose-problems             1       1
rose-winter               1       1
orchid-pests              5       1
orchid-problems           1       1
hydrangea-water           1       1
hydrangea-pests           3       1
hydrangea-winter          1       1
dahlia-winter             1       1
dahlia-pests              1       1
strawberry-problems       1       1
basil-problems            1       1
basil-winter              1       1
cucumber-water            1       1
cucumber-pests            1       1
first:      search 17 / 20,  rerank top 10 19 / 20
in top 3:   search 19 / 20,  rerank top 10 19 / 20


## 8. How many candidates? Quality for each k

The reranker can only reorder what the search hands it. For k = 5, 10, 20 and 40 the cell counts how often the
answer is among the k candidates, and how often it is in the top three after reranking.

**Look for:** k = 5 already holds all 20 answers (the lowest sat exactly 5th, so there is no margin). In the
top three after reranking: 20 / 20 for k = 5, 19 / 20 for k = 10, 20 and 40. More candidates did not give better answers on these questions.

In [8]:
print(" k   answer among the k   in top 3 after rerank")
for k in (5, 10, 20, 40):
    among = sum(r["gold"] in r["order"][:k] for r in results)
    top3 = sum((place_after(r, k) or 99) <= 3 for r in results)
    print(f"{k:2}   {among:6} / 20           {top3:6} / 20")

 k   answer among the k   in top 3 after rerank
 5       20 / 20               20 / 20
10       20 / 20               19 / 20
20       20 / 20               19 / 20
40       20 / 20               19 / 20


## 9. How long does each step take?

The search step encodes one question and compares it with the 45 stored vectors. The rerank step runs the
cross-encoder on k pairs. Each time is the median of 15 runs after 3 warm-up runs, for the orchid question.

**Look for:** the rerank time grows with k: k = 40 takes several times as long as k = 5. The search runs the
model once, on the question; the reranker runs it once per pair. With only 45 chunks the search costs about as
much as reranking a few pairs; in a shop with thousands of chunks the search stays fast, while the reranker's
cost is set by k. Your times will not match the lesson's (another library and another CPU), and they jump a
little from run to run.

In [9]:
import time, statistics

def median_ms(fn, runs=15, warm=3):
    for _ in range(warm):
        fn()
    times = []
    for _ in range(runs):
        t = time.perf_counter(); fn(); times.append(time.perf_counter() - t)
    return statistics.median(times) * 1000

def search_step():
    v = search.encode([question])
    return search.similarity(v, chunk_vectors)[0].argsort(descending=True)

order = search_step().tolist()
print(f"search (all 45 chunks): {median_ms(search_step):7.1f} ms")
for k in (5, 10, 20, 40):
    pairs = [(question, care_chunks[i]) for i in order[:k]]
    print(f"rerank k = {k:2}:          {median_ms(lambda: reranker.predict(pairs)):7.1f} ms")

search (all 45 chunks):     6.2 ms


rerank k =  5:             15.2 ms


rerank k = 10:             24.9 ms


rerank k = 20:             39.8 ms


rerank k = 40:             86.1 ms


## 10. Broken on purpose: whole sheets, `max_length=128`

A colleague "simplified" the assistant: no sections, each plant's whole sheet is one passage, all nine go to the
reranker, and `max_length=128` to make it faster. The answer to this question is in the rose sheet (black spot,
in its last section). The cell prints how many tokens each pair has, and how many the model reads.

**Look for:** every pair has more than 128 tokens, so every sheet is cut, with no warning. The rose sheet is
**8th** of 9. The last words the model read of the rose sheet stop in its winter section, before "problems".

In [10]:
plants = list(dict.fromkeys(i.split("-")[0] for i in ids))
whole = {p: " ".join(SHEETS[c] for c in ids if c.split("-")[0] == p) for p in plants}
sheet_question = "A customer brought in a leaf with dark blotches. The leaves turn yellow and drop from midsummer. What is it?"
pairs = [(sheet_question, whole[p]) for p in plants]

def rerank_sheets(max_length):
    model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2", max_length=max_length)
    sheet_scores = model.predict(pairs)
    tokens = [len(model.tokenizer(q, p)["input_ids"]) for q, p in pairs]
    print(f"max_length={max_length}  (sheet, score, tokens in pair, tokens read)")
    for place, n in enumerate(sorted(range(len(plants)), key=lambda n: -sheet_scores[n]), 1):
        print(f"  {place}  {plants[n]:<11} {sheet_scores[n]:8.4f}  {tokens[n]:4}  {min(tokens[n], max_length):4}")
    kept = model.tokenizer(sheet_question, whole["rose"], truncation=True, max_length=max_length)["input_ids"]
    print("last words read of the rose sheet: ...", model.tokenizer.decode(kept, skip_special_tokens=True)[-160:])

rerank_sheets(128)

max_length=128  (sheet, score, tokens in pair, tokens read)
  1  lemon        -7.8567   215   128
  2  dahlia       -8.7932   200   128
  3  cucumber     -9.2369   190   128
  4  orchid       -9.6109   193   128
  5  hydrangea    -9.8018   220   128
  6  strawberry  -10.0939   166   128
  7  basil       -10.1603   173   128
  8  rose        -10.5658   190   128
  9  tomato      -10.6911   202   128
last words read of the rose sheet: ... se : pests. greenfly cluster on soft shoot tips and flower buds in spring. squash small groups by hand and encourage ladybirds. rose : winter and pruning. prune


## 11. The fix: let the model read the whole passage

The same run with the model's own limit, 512 tokens. Every pair now fits. (The real fix in the assistant is
the one the lesson uses: short passages, one section each, so every pair fits.)

**Look for:** the rose sheet is now **1st**, and the last words read include the black spot section.

In [11]:
rerank_sheets(512)

max_length=512  (sheet, score, tokens in pair, tokens read)
  1  rose         -0.7098   190   190
  2  lemon        -5.1603   215   215
  3  dahlia       -8.1850   200   200
  4  tomato       -8.4107   202   202
  5  cucumber     -8.5456   190   190
  6  orchid       -8.6149   193   193
  7  basil        -9.2470   173   173
  8  hydrangea    -9.4976   220   220
  9  strawberry  -10.1614   166   166
last words read of the rose sheet: ... ot is a fungus that marks the leaves with dark blotches ; the leaves turn yellow and fall from midsummer. clear up fallen leaves and choose resistant varieties.


## Now change it

Three small experiments. Each cell has one line marked `# <- change this`. Change it and run the cell again.

### Experiment 1: your own question

The cell compares the search's top three with what `find_chunks` returns for one question. Write any question a
customer might ask about these nine plants.

**What should change:** for the tomato question below, both put `tomato-problems` (blight) first. But the
reranker's second and third chunks are about other plants (lemon pests, rose problems): it judges each pair on
its own and does not care which plant a chunk belongs to. Try a question in your own words, for example about
an orchid that has not flowered for a year, and see what moves up and what drops out.

In [12]:
my_question = "My tomato leaves are covered in brown spots after the rain."  # <- change this
s = search.similarity(search.encode([my_question]), chunk_vectors)[0]
print("search top 3:     ", [ids[i] for i in s.argsort(descending=True)[:3].tolist()])
print("find_chunks top 3:", [ids[care_chunks.index(t)] for t in find_chunks(my_question)])

search top 3:      ['tomato-problems', 'tomato-water', 'tomato-winter']
find_chunks top 3: ['tomato-problems', 'lemon-pests', 'rose-problems']


### Experiment 2: too few candidates

The cell runs `find_chunks` on the orchid question with a different k.

**What should change:** with k = 10 the answer, `orchid-pests`, comes back first. With k = 4 it is gone: the
search put it 5th, so it is never a candidate, and the reranker cannot rank a chunk it never sees.

In [13]:
k = 10  # <- change this (try 4)
print("k =", k, "->", [ids[care_chunks.index(t)] for t in find_chunks(question, k=k)])

k = 10 -> ['orchid-pests', 'orchid-problems', 'dahlia-winter']


### Experiment 3: how much must the reranker read?

The cell reranks the nine whole sheets again with a `max_length` you choose.

**What should change:** with 192 the whole rose sheet fits (it has 190 tokens), so the model reads the black spot
section and the rose sheet is 1st, with the same score as with 512 (−0.71). With 160 it is still cut before
"problems" and stays near the bottom. The model must read the part of the passage that holds the answer.

In [14]:
max_length = 128  # <- change this (try 192 or 256)
rerank_sheets(max_length)

max_length=128  (sheet, score, tokens in pair, tokens read)
  1  lemon        -7.8567   215   128
  2  dahlia       -8.7932   200   128
  3  cucumber     -9.2369   190   128
  4  orchid       -9.6109   193   128
  5  hydrangea    -9.8018   220   128
  6  strawberry  -10.0939   166   128
  7  basil       -10.1603   173   128
  8  rose        -10.5658   190   128
  9  tomato      -10.6911   202   128
last words read of the rose sheet: ... se : pests. greenfly cluster on soft shoot tips and flower buds in spring. squash small groups by hand and encourage ladybirds. rose : winter and pruning. prune


## What you should see

- Step 3: the model card's two Berlin pairs score about 8.61 and −4.32, so your setup matches the lesson's.
- Step 4: the search puts `orchid-pests` 5th. Step 5: after reranking the top 10 it is 1st.
- Step 5: every reranker score for the orchid question's top 10 is below zero, yet the order is right.
- Step 9: the rerank time grows with k: k = 40 takes several times as long as k = 5.
- Steps 10 and 11: with `max_length=128` on whole sheets the rose sheet is 8th of 9; with 512 it is 1st.